In [1]:
%load_ext autoreload
%autoreload 2
%load_ext dotenv
%dotenv

In [2]:
import os

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"  # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [3]:
### Shortcut for package import
from pkgimp import *

from nb2p import database, config
from nb2p.notebook import Notebook
from nb2p import astparse
from nb2p.dfgtree import preprocess

In [4]:
# Load model directly
from tokenizers import Tokenizer
from transformers import RobertaConfig, RobertaForSequenceClassification
# from model import Model
from nb2p.dfgtree.compressor.model import Model

In [5]:
torch.set_num_threads(1)
torch.get_num_threads()

1

In [6]:
def get_xs_model(base_dir: str):
    config = RobertaConfig.from_pretrained("microsoft/graphcodebert-base")
    config.num_attention_heads = 8
    config.hidden_size = 96
    config.intermediate_size = 64
    config.vocab_size = 1000
    config.num_hidden_layers = 12
    config.hidden_dropout_prob = 0.2

    tokenizer_path = os.path.join(base_dir, f"BPE_{str(config.vocab_size)}.json")
    tokenizer = Tokenizer.from_file(tokenizer_path)

    model = Model(RobertaForSequenceClassification(config=config), config, tokenizer)

    model_dir = os.path.join(base_dir, "GraphCodeBERT/clone_detection/checkpoint", "3", "model.bin")
    model.load_state_dict(torch.load(model_dir))

    return model, tokenizer

In [7]:
DEVICE = torch.device("cuda")

model, tokenizer = get_xs_model(
    os.path.join(
        os.environ["NB2P_PREFIX"],
        "nb2p/dfgtree/compressor",
    )
)
model.eval()
model = model.to(DEVICE)

DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): huggingface.co:443


DEBUG:urllib3.connectionpool:https://huggingface.co:443 "HEAD /microsoft/graphcodebert-base/resolve/main/config.json HTTP/11" 200 0


/tmp/ipykernel_557282/2611845937.py:16: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_dir))


In [8]:
from nb2p.dfgtree.dfg import CodeEncodingBuilder

encoding_builder = CodeEncodingBuilder(tokenizer, model)
encoding_builder

<CodeEncodingBuilder device=cuda>

In [9]:
_DEBUG_CODE_STR = """i = 1
from os import (
    path,
    environ
)
import time
def foo():
    if bar:
        baz()
        def qux():
            quux()
def qux(one: str, two, three):
    quux()
qux()"""

inp = encoding_builder.make_input(_DEBUG_CODE_STR)

In [10]:
out = encoding_builder.get_encoding(inp)
out.shape, out

(torch.Size([1, 96]),
 tensor([[ 0.0459,  0.5318,  0.5849,  0.3224, -0.0777,  0.1114, -0.1325,  0.5421,
          -0.2223,  0.5152,  0.2335, -0.3315, -1.0782,  0.3075,  1.2094, -1.3458,
           1.4802,  0.9046,  0.4666, -0.3980, -0.3756,  0.1517, -1.9829, -0.3983,
           0.6725,  1.2005, -1.0401, -0.0304,  0.7750,  0.0961,  0.7896, -0.7715,
          -0.9073,  0.0583, -0.1441, -0.2353, -0.2811, -0.4528,  0.6436, -1.0412,
           0.7175,  0.1609,  2.9971, -0.4830, -0.7275, -0.6534, -0.1935, -0.7905,
           0.4232,  0.1421,  0.7500,  1.1589,  0.4597,  1.3842,  0.2456, -0.6045,
           0.0459, -0.6196, -0.1072, -1.5798, -0.4606, -1.8542, -0.0883,  0.4313,
          -0.1825,  1.7083,  0.2438,  1.4715,  0.5172,  1.4528,  1.0124,  0.7268,
          -0.9141, -1.2062,  0.0710, -0.0667, -0.6395,  0.1283, -0.5220, -0.3555,
          -1.1694,  0.8614,  0.4743, -0.9364, -1.8166, -0.1661,  0.8547,  0.4033,
           1.1883, -1.1638, -1.8919, -0.6588,  0.0107, -1.1938,  0.2290,  0.

In [11]:
### Select dataset to process

DATASET_NAME = "distilkaggle"
# DATASET_NAME = "pmbf"

In [12]:
DIRS = config.dirs(dataset_name=DATASET_NAME)
DIRS.makedirs()

making dirs: /ssd/haotian/scs/distilkaggle/processed-unixcoder-ast
making dirs: /ssd/haotian/scs/distilkaggle
making dirs: /ssd/haotian/scs/distilkaggle/processed-unixcoder-full
making dirs: /ssd/haotian/scs/distilkaggle/dfgtree
making dirs: /ssd/haotian/scs/distilkaggle/processed-unixcoder-eda
making dirs: /ssd/haotian/scs/distilkaggle/logs/full
making dirs: /ssd/haotian/scs/distilkaggle/models/full
making dirs: /ssd/haotian/scs/distilkaggle/ipynb


In [13]:
parser, lang = astparse.parser()

db, client = database.connect(dataset_name=DATASET_NAME, verbose=True)

Pinged to database nb2p-dk. You successfully connected to MongoDB!


In [14]:
a_notebook_data = database.get_notebooks(db, {"prompted": True, "segments.6": {"$exists": True}}, include_segments=True).next()
print(a_notebook_data['_id'])
a_notebook = Notebook.from_db_result(a_notebook_data)
a_notebook

66f3ce8e975730f2dded4f49


<Notebook(11 segments, 0 markdown cells, 15 code cells)>

## Test embedding

In [15]:
preproc_result = preprocess(a_notebook, parser, lang)
for s in preproc_result['imports']:
    print(s)
    print("___END___")

print("__SEGMENTS__")

for s in preproc_result['segments']:
    print(s)
    print("___END___")

import numpy as np
___END___
import matplotlib.pyplot as plt
___END___
from sklearn.metrics import accuracy_score
___END___
import sklearn
___END___
import sklearn.datasets
___END___
import warnings
___END___
import sklearn.linear_model
___END___
import sklearn.tree
___END___
import sklearn.svm
___END___
from sklearn.linear_model import LinearRegression
___END___
from sklearn.metrics import r2_score
___END___
from sklearn.metrics import mean_squared_error
___END___
from sklearn.preprocessing import PolynomialFeatures
___END___
from sklearn.pipeline import Pipeline
___END___
from sklearn import cross_validation
___END___
import timeit
___END___
__SEGMENTS__
%matplotlib inline
___END___
%matplotlib inline
x = np.linspace(0, 0.5, 100)
plt.plot( x, 0.7 - 0.5*x + 0.3*np.exp(-x*20), label = "Overfitted model")
plt.plot( x, 0.9 - 0.5*x, label = "Non-Overfitted model")
plt.plot( x, 0.6 - 0.5*x, label = "Just Bad model")
axes = plt.gca()
axes.set_ylim([0, 1.1])
plt.legend(loc=3)
plt.suptitle("E

## List embedding data

In [16]:
ids = sorted(list(
    map(
        lambda x: x["_id"],
        db.notebooksegments.find(
            {"n_ast_children_of_segments": {"$lt": 256}, "selected": True},
            {"_id": 1},
        ),
    )
))
len(ids), ids[:3]

(1020,
 [ObjectId('66f3ce8e975730f2dded4f33'),
  ObjectId('66f3ce8e975730f2dded4f7f'),
  ObjectId('66f3ce8e975730f2dded5051')])

In [17]:
# X_train_ids, X_test_ids = train_test_split(ids, test_size=0.2, random_state=42)
# len(X_train_ids), len(X_test_ids)
X_train_ids = ids

## Save to ChromaDB

In [18]:
import logging
import http.client

for log in ["urllib3", "httpx", "httpcore.http11", "httpcore.connection"]:
    requests_log = logging.getLogger(log)
    requests_log.setLevel(logging.WARNING)
    requests_log.propagate = False

http.client.HTTPConnection.debuglevel = -1
logging.basicConfig(level=logging.WARNING)

In [19]:
# chroma_client.delete_collection(name=DATASET_NAME)

In [20]:
from chromadb import Documents, EmbeddingFunction, Embeddings
class GCBEmbeddingFunction(EmbeddingFunction):
    def __call__(self, input: Documents) -> Embeddings:
        result = []
        for t in input:
            inp = encoding_builder.make_input(t)
            out = encoding_builder.get_encoding(inp).detach().cpu()[0]
            result.append(out)

        return result

In [21]:
pp = preprocess(a_notebook, parser, lang)
test_code = "\n".join(pp['segments'])
print(GCBEmbeddingFunction()([test_code]))

[array([ 0.12361665,  1.2844385 , -0.24898377,  0.19363642,  0.03152184,
        1.1733993 ,  0.8906153 ,  0.88502043,  0.5121522 ,  0.0774403 ,
       -0.98009187,  0.32505998, -1.5754937 ,  0.08705012,  0.7727103 ,
       -0.6896204 ,  0.5588072 , -0.7494541 , -0.43321094,  0.61782706,
       -0.10283504, -0.2080429 , -0.8166272 , -0.93438023,  1.2658952 ,
       -0.08214892, -0.1208718 ,  0.50694376,  0.5272532 ,  0.67568016,
        0.46725246,  0.35253608,  0.23149791,  0.22553782, -0.47276402,
        0.14455307,  0.4946561 ,  0.4829587 , -0.27156416,  0.46768817,
       -0.37309566, -0.8913736 ,  0.67800516,  1.2979609 ,  0.8855497 ,
       -1.0355011 , -0.4656542 , -0.25423613, -0.10804329,  0.5884143 ,
       -0.6835879 ,  0.92024755,  0.49606085,  0.961329  , -0.8309078 ,
       -0.905231  , -0.43502817, -0.9764704 ,  0.5405215 , -0.44677973,
       -0.6719441 , -0.29616782, -0.93037474,  1.1805286 , -0.36093074,
        0.42492667,  0.03541496, -0.2836065 , -0.71632475,  0.4

In [22]:
import chromadb
chromadb.logger.setLevel(logging.WARNING)

chroma_client = chromadb.HttpClient(port=8000)
collection = chroma_client.get_or_create_collection(name=DATASET_NAME, embedding_function=GCBEmbeddingFunction())
collection

Collection(id=f0c453ec-280b-45b5-b2c8-7f9a3b7c2b4f, name=distilkaggle2)

In [23]:
collection.get(
	ids=["66f3ce8e975730f2dded4f39"],
    include=["metadatas"]
)

{'ids': [],
 'embeddings': None,
 'metadatas': [],
 'documents': None,
 'data': None,
 'uris': None,
 'included': ['metadatas']}

In [ ]:
from db import to_chromadb

to_chromadb(db, collection, encoding_builder, parser, lang, X_train_ids) is None

In [24]:
collection.query(
    query_texts=[test_code]
)

{'ids': [['67c8071dde3903ec14ca30c2',
   '67c8071dde3903ec14ca8fe2',
   '67c8071dde3903ec14cb9546',
   '67c8071dde3903ec14ca18b3',
   '67c8071dde3903ec14ca6178',
   '67c8071dde3903ec14ca617f',
   '67c8071dde3903ec14ca4c08',
   '67c8071cde3903ec14c7f152',
   '67c8071cde3903ec14c8210c',
   '67c8071cde3903ec14c7ffb8']],
 'distances': [[0.4156107008457184,
   0.4175318479537964,
   0.447146475315094,
   0.454775869846344,
   0.454775869846344,
   0.454775869846344,
   0.45863187313079834,
   0.46950745582580566,
   0.46993693709373474,
   0.4783244729042053]],
 'embeddings': None,
 'metadatas': [[{'segment_ends': '[0, 2, 6, 26, 86, 93, 101, 102, 104, 106, 107, 109, 111, 112, 114, 119, 121, 125, 127, 131, 133, 142]'},
   {'segment_ends': '[1, 2, 5, 18, 20, 26, 28, 30, 32, 37, 42]'},
   {'segment_ends': '[3, 4, 25]'},
   {'segment_ends': '[1, 19, 24, 27, 33, 43, 52, 54, 57]'},
   {'segment_ends': '[1, 19, 24, 27, 34, 46, 55, 57, 60]'},
   {'segment_ends': '[1, 19, 24, 27, 34, 36, 48, 53, 66,

In [25]:
dsp_db, client = database.connect(dataset_name="dspipeline2", verbose=True)

Pinged to database nb2p-dsp2. You successfully connected to MongoDB!


In [28]:
test_ids = sorted(list(
    map(
        lambda x: x["_id"],
        dsp_db.notebooksegments.find(
            {},
            {"_id": 1},
        ),
    )
))
len(test_ids), ids[:3]

(92,
 [ObjectId('67c8071cde3903ec14c7e2f8'),
  ObjectId('67c8071cde3903ec14c7e2fa'),
  ObjectId('67c8071cde3903ec14c7e2fc')])

In [29]:
import logging

logging.getLogger().setLevel(logging.WARN)

for i, nb_id in tqdm(enumerate(test_ids), total=len(test_ids)):
    nb_data = dsp_db.notebooksegments.find_one({"_id": nb_id})
    if nb_data is None:
        ignored.append(nb_id)
        print(f"WARN  no such notebook: {nb_id}")
        continue

    nb_id = str(nb_id)

    nb = Notebook.from_db_result(nb_data)
    try:
        pr = preprocess(nb, parser, lang)
    except Exception as e:
        ignored.append(nb_id)
        print(nb_id, e)
        continue
    
    code = "\n".join(pr['segments'])
    segment_ends = nb_data['encoding']['segment_ends']
    
    collection.delete(
        where={'segment_ends': str(segment_ends)}
    )

100%|██████████████████████████████████████████████████████████████████| 92/92 [00:04<00:00, 18.95it/s]
